# Translate Corpus: 10-row chat JSONL example

This notebook demonstrates the current Nemotron `translate/translation` step structure. It does not import Curator internals directly. Instead, it writes a small chat JSONL corpus, derives a working config from `src/nemotron/steps/translate/translation/config/default.yaml`, and runs the public CLI surface: `nemotron steps translation`.

Output rows preserve the OpenAI-style chat schema and (optionally) carry FAITH translation-quality scores.

## What this notebook creates

- `assets/sample_chat.jsonl`: 10 chat rows with `messages.*.content` to translate.
- `config/en_to_hi.yaml`: a working translation config derived from the packaged step default.
- `outputs/en_to_hi/`: translated JSONL shards with FAITH scores when enabled.

The run cell defaults to command preview mode so the notebook does not accidentally spend model quota.

## 1. Locate the repository and translation step assets

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import yaml


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/nemotron/steps/translate/translation").is_dir():
            return candidate
    raise RuntimeError("Could not find the Nemotron repository root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
EXAMPLE_DIR = REPO_ROOT / "use-case-examples" / "translate-corpus"
STEP_CONFIG_DIR = SRC_DIR / "nemotron" / "steps" / "translate" / "translation" / "config"
STEP_DEFAULT_CONFIG = STEP_CONFIG_DIR / "default.yaml"

ASSETS_DIR = EXAMPLE_DIR / "assets"
CONFIG_DIR = EXAMPLE_DIR / "config"
OUTPUT_DIR = EXAMPLE_DIR / "outputs" / "en_to_hi"
INPUT_JSONL = ASSETS_DIR / "sample_chat.jsonl"
WORKING_CONFIG = CONFIG_DIR / "en_to_hi.yaml"

for path in (ASSETS_DIR, CONFIG_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repository root:", REPO_ROOT)
print("Packaged translation config:", STEP_DEFAULT_CONFIG)
print("Example input JSONL:", INPUT_JSONL)
print("Working config:", WORKING_CONFIG)
print("Output directory:", OUTPUT_DIR)

## 2. Credentials

The default translation config uses NVIDIA-hosted OpenAI-compatible endpoints. Export `NGC_API_KEY` or `NVIDIA_API_KEY` before starting Jupyter, or set them in this cell for the current notebook process. FAITH scoring uses the same endpoint, so the key is required when `faith_eval.enabled=true` even if `backend != llm`.

In [ ]:
NGC_API_KEY = None
NVIDIA_API_KEY = None

if NGC_API_KEY:
    os.environ["NGC_API_KEY"] = NGC_API_KEY
    os.environ.setdefault("NVIDIA_API_KEY", NGC_API_KEY)

if NVIDIA_API_KEY:
    os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
    os.environ.setdefault("NGC_API_KEY", NVIDIA_API_KEY)

print("NGC_API_KEY set:", bool(os.environ.get("NGC_API_KEY")))
print("NVIDIA_API_KEY set:", bool(os.environ.get("NVIDIA_API_KEY")))

## 3. Write a 10-row chat JSONL corpus

Each row is OpenAI-style: a `messages` list of `{role, content}` pairs. The translation step reads `messages.*.content` and reconstructs the messages list with translated content.

In [ ]:
sample_rows = [
    {"messages": [
        {"role": "user", "content": "What is the capital of France?"},
        {"role": "assistant", "content": "The capital of France is Paris."},
    ]},
    {"messages": [
        {"role": "user", "content": "Convert 25 degrees Celsius to Fahrenheit."},
        {"role": "assistant", "content": "25 degrees Celsius is equivalent to 77 degrees Fahrenheit."},
    ]},
    {"messages": [
        {"role": "user", "content": "List three colors in the rainbow."},
        {"role": "assistant", "content": "Three colors in the rainbow are red, green, and blue."},
    ]},
    {"messages": [
        {"role": "user", "content": "Who wrote the play Hamlet?"},
        {"role": "assistant", "content": "Hamlet was written by William Shakespeare."},
    ]},
    {"messages": [
        {"role": "user", "content": "What is the largest planet in the solar system?"},
        {"role": "assistant", "content": "Jupiter is the largest planet in the solar system."},
    ]},
    {"messages": [
        {"role": "user", "content": "Define photosynthesis in one sentence."},
        {"role": "assistant", "content": "Photosynthesis is the process by which plants convert light energy into chemical energy."},
    ]},
    {"messages": [
        {"role": "user", "content": "What is the boiling point of water at sea level?"},
        {"role": "assistant", "content": "Water boils at 100 degrees Celsius at sea level."},
    ]},
    {"messages": [
        {"role": "user", "content": "Name the inventor of the telephone."},
        {"role": "assistant", "content": "Alexander Graham Bell is credited with inventing the telephone."},
    ]},
    {"messages": [
        {"role": "user", "content": "What gas do plants absorb from the air?"},
        {"role": "assistant", "content": "Plants absorb carbon dioxide from the air for photosynthesis."},
    ]},
    {"messages": [
        {"role": "user", "content": "How many continents are there on Earth?"},
        {"role": "assistant", "content": "There are seven continents on Earth."},
    ]},
]

with INPUT_JSONL.open("w", encoding="utf-8") as f:
    for row in sample_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Wrote {len(sample_rows)} rows to {INPUT_JSONL}")
print(INPUT_JSONL.read_text(encoding="utf-8").splitlines()[0])

## 4. Write the example config from the packaged translation config

This keeps the notebook consistent with the framework: the packaged config is the template, and the example only overrides paths, language codes, and FAITH/output toggles. Setting `text_field=messages.*.content` plus `reconstruct_messages=true` is the standard rule for OpenAI-style chat JSONL.

In [ ]:
cfg = yaml.safe_load(STEP_DEFAULT_CONFIG.read_text(encoding="utf-8"))

cfg.update(
    {
        "input_path": str(INPUT_JSONL),
        "output_dir": str(OUTPUT_DIR),
        "source_language": "en",
        "target_language": "hi",
        "input_format": "jsonl",
        "output_format": "jsonl",
        "text_field": "messages.*.content",
        "output_mode": "both",
        "reconstruct_messages": True,
        "max_concurrent_requests": 4,
    }
)
cfg.setdefault("server", {})
cfg["server"].update(
    {
        "url": cfg["server"].get("url", "https://integrate.api.nvidia.com/v1"),
        "api_key_env": cfg["server"].get("api_key_env", "NVIDIA_API_KEY"),
    }
)
if not cfg["server"].get("model"):
    cfg["server"]["model"] = "openai/gpt-oss-120b"
cfg.setdefault("faith_eval", {})
cfg["faith_eval"].update({
    "enabled": True,
    "threshold": 2.5,
    "filter_enabled": False,
})

WORKING_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print(WORKING_CONFIG.read_text(encoding="utf-8"))

## 5. Run translation through the Nemotron CLI

In [ ]:
def run_translation(*args: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    env = os.environ.copy()
    existing_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = str(SRC_DIR) + (os.pathsep + existing_pythonpath if existing_pythonpath else "")
    cmd = [sys.executable, "-m", "nemotron", "steps", "translation", *map(str, args)]
    print("$", " ".join(cmd))
    return subprocess.run(cmd, cwd=REPO_ROOT, env=env, text=True, check=check)


RUN_TRANSLATE = False

translate_args = ["-c", str(WORKING_CONFIG)]

if RUN_TRANSLATE:
    if not (os.environ.get("NGC_API_KEY") or os.environ.get("NVIDIA_API_KEY")):
        raise RuntimeError("NGC_API_KEY or NVIDIA_API_KEY is required before running translation.")
    run_translation(*translate_args)
else:
    print("Preview only. Command to run:")
    print("python -m nemotron steps translation", " ".join(translate_args))

## 6. Preview translated output and FAITH scores

Curator's writer emits one or more `*.jsonl` shards under `output_dir`. The cell below loads up to the first 10 rows and shows translated content plus FAITH metadata when present.

In [ ]:
shards = sorted(OUTPUT_DIR.rglob("*.jsonl"))
print("Found shards:", [str(p.relative_to(OUTPUT_DIR)) for p in shards])

if shards:
    rows = []
    for shard in shards:
        with shard.open("r", encoding="utf-8") as f:
            for line in f:
                rows.append(json.loads(line))
                if len(rows) >= 10:
                    break
        if len(rows) >= 10:
            break
    for i, row in enumerate(rows):
        print(f"-- row {i} --")
        original = row.get("messages", [])
        translated = row.get("translated_messages", [])
        for src, tgt in zip(original, translated):
            print(f"  {src.get('role')}: {src.get('content')}")
            print(f"    \u2192 {tgt.get('content')}")
        if "faith_score_avg" in row:
            print(f"  FAITH avg: {row['faith_score_avg']}")
else:
    print("No translated shards yet. Set RUN_TRANSLATE=True and re-run cell 5.")